# Part 1: Data Acquisition
## Downloading VEGFR2 IC50 Data from ChEMBL

This notebook downloads bioactivity data for VEGFR2 (CHEMBL279) from the ChEMBL database.

**Target:** VEGFR2 (VEGF Receptor 2)  
**Data Source:** ChEMBL279  
**Measurement:** IC50 (nM)

In [ ]:
# @title 1. Install Dependencies
# pip install rdkit pandas numpy

In [ ]:
# @title 2. Download Data from ChEMBL
import urllib.request
import json
import csv
import pandas as pd
from pathlib import Path

# ChEMBL API configuration
BASE_URL = "https://www.ebi.ac.uk/chembl/api/data"
TARGET_ID = "CHEMBL279"

def download_chembl_data(target_id, output_path):
    """Download IC50 data for a target from ChEMBL."""
    rows = []
    offset = 0
    page_size = 1000
    
    print(f"Downloading VEGFR2 (CHEMBL279) IC50 data...")
    
    while True:
        url = f"{BASE_URL}/activity.json?target_chembl_id={target_id}&standard_type=IC50&limit={page_size}&offset={offset}"
        try:
            with urllib.request.urlopen(url) as resp:
                data = json.load(resp)
        except Exception as e:
            print(f"  Error at offset {offset}: {e}")
            break
        
        activities = data.get("activities", [])
        if not activities:
            break
        
        for act in activities:
            if act.get("standard_relation") != "=":
                continue
            smiles = act.get("canonical_smiles")
            val = act.get("standard_value")
            if not smiles or val is None:
                continue
            try:
                ic50 = float(val)
                if ic50 <= 0:
                    continue
            except (ValueError, TypeError):
                continue
            rows.append({"smiles": smiles, "ic50_nM": ic50})
        
        print(f"  Offset {offset}: {len(rows)} compounds so far")
        
        if len(activities) < page_size:
            break
        offset += page_size
    
    # Save to CSV
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    df = pd.DataFrame(rows)
    df.to_csv(output_path, index=False)
    print(f"\nSaved {len(df)} compounds to {output_path}")
    return df

# Download
df = download_chembl_data(TARGET_ID, "data/chembl_vegfr2.csv")

In [ ]:
# @title 3. Data Statistics
print(f"Dataset Statistics:")
print(f"  Total compounds: {len(df)}")
print(f"  Unique SMILES: {df['smiles'].nunique()}")
print(f"  IC50 range: {df['ic50_nM'].min():.1f} - {df['ic50_nM'].max():.1f} nM")
print(f"  IC50 median: {df['ic50_nM'].median():.1f} nM")

# Activity distribution (500 nM threshold)
df['active'] = (df['ic50_nM'] < 500).astype(int)
print(f"\nActivity (at 500 nM threshold):")
print(f"  Active (IC50 < 500 nM): {df['active'].sum()} ({df['active'].mean():.1%})")
print(f"  Inactive (IC50 >= 500 nM): {(1-df['active']).sum()} ({1-df['active'].mean():.1%})")

df.head(10)

In [ ]:
# @title 4. IC50 Distribution
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Raw IC50
axes[0].hist(df['ic50_nM'], bins=50, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('IC50 (nM)')
axes[0].set_ylabel('Count')
axes[0].set_title('IC50 Distribution')
axes[0].axvline(x=500, color='red', linestyle='--', label='500 nM threshold')
axes[0].legend()

# Log IC50
axes[1].hist(np.log10(df['ic50_nM']), bins=50, edgecolor='black', alpha=0.7, color='green')
axes[1].set_xlabel('log10(IC50)')
axes[1].set_ylabel('Count')
axes[1].set_title('Log IC50 Distribution')
axes[1].axvline(x=np.log10(500), color='red', linestyle='--', label='500 nM threshold')
axes[1].legend()

plt.tight_layout()
plt.savefig('images/ic50_distribution.png', dpi=150, bbox_inches='tight')
plt.show()